# Qualifire — Complex types in profiling, shape, and pattern

Focused walkthrough of how the library handles **non-scalar columns**
(`array`, `map`, `struct`; pandas `object` holding dicts / lists). The
three places that interact with column types treat them differently:

| Place | Default behaviour | Knob to override |
|---|---|---|
| Profiling (`ProfilingCollectionConfig`) | Spark `ArrayType` / `MapType` / `StructType` / `BinaryType` are **silently skipped** (debug-logged). Pandas `object` columns are classified as `string` and profiled best-effort. | None — surface complex columns as plain string columns upstream if you need them profiled. |
| Shape (`shape_check` / `IsolationForest`) | Unhashable (dict / list) → **stringify + LabelEncode**. Other complex dtypes same. | `drop_complex=True` drops them instead. |
| Pattern (`pattern_check` / `RandomForest`) | Same as shape (shares `_encoding.py`). | `drop_complex=True` drops them. |

Both `_encoding.py` paths fall through to the same encoder, so the
two validators stay consistent — a `drop_complex` flag set on shape
and the same flag set on pattern produce the same feature matrix
from the same DataFrame.

If you need pyspark + JDK setup help, run
[`pyspark_setup.ipynb`](pyspark_setup.ipynb) first. This notebook
assumes the kernel already has `pyspark`, `qualifire`, and (for the
shape / pattern cells) `scikit-learn` + `shap` available.

## Setup

Spark session + Qualifire wired to an in-memory SQLite system table.
Keeps the cell count tight — full setup commentary lives in
`notebook_overall.ipynb`.

In [ ]:
import os, sys, re, glob, subprocess, tempfile
from pathlib import Path

# Spark 3.5 requires Java 8/11/17. Refuse anything else (Java 18+ removes
# Subject.getSubject which Hadoop 3.3.4 calls during SparkContext init).
ACCEPTABLE = (8, 11, 17)

def _java_major(java_bin):
    try:
        out = subprocess.check_output([java_bin, '-version'], stderr=subprocess.STDOUT, text=True)
    except Exception:
        return None
    m = re.search(r'version "([^"]+)"', out)
    if not m: return None
    parts = m.group(1).split('.')
    return 8 if parts[:2] == ['1', '8'] else int(parts[0])

def _pick_jdk():
    for pat in ['/Library/Java/JavaVirtualMachines/*/Contents/Home',
                os.path.expanduser('~/Library/Java/JavaVirtualMachines/*/Contents/Home')]:
        for h in sorted(glob.glob(pat)):
            if 'JavaAppletPlugin' in h:
                continue
            jb = os.path.join(h, 'bin', 'java')
            if os.path.isfile(jb) and _java_major(jb) in ACCEPTABLE:
                return h, _java_major(jb)
    return None, None

java_home, mj = _pick_jdk()
if java_home:
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = f'{java_home}/bin:' + os.environ.get('PATH', '')
    print(f'JAVA_HOME : {java_home}  (Java {mj})')
else:
    print('No Java 8/11/17 JDK found — install one (e.g. brew install --cask temurin@8)')

# Scrub Downloads-rooted entries from sys.path (macOS Privacy gotcha).
sys.path[:] = [p for p in sys.path if '/Downloads/' not in p]
os.environ.pop('PYTHONPATH', None)
for v in ('SPARK_HOME',):
    if '/Downloads/' in os.environ.get(v, ''):
        os.environ.pop(v, None)
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

WORK_DIR = Path(tempfile.mkdtemp(prefix='qf_complex_'))
WAREHOUSE = WORK_DIR / 'warehouse'
SYSTEM_DB = WORK_DIR / 'qf.sqlite'
print('WORK_DIR :', WORK_DIR)

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

if SparkContext._gateway is not None:
    try: SparkContext._gateway.shutdown()
    except Exception: pass
    SparkContext._gateway = SparkContext._jvm = SparkContext._active_spark_context = None

metastore_url = f'jdbc:derby:;databaseName={WORK_DIR}/metastore_db;create=true'
spark = (
    SparkSession.builder.master('local[*]').appName('qf-complex-types')
    .config('spark.sql.warehouse.dir', str(WAREHOUSE))
    .config('javax.jdo.option.ConnectionURL', metastore_url)
    .config('spark.driver.extraJavaOptions', f'-Dderby.system.home={WORK_DIR}')
    .enableHiveSupport().getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark JVM:', spark.sparkContext._jvm.System.getProperty('java.version'))

from qualifire.api import Qualifire
from qualifire.backends.spark_backend import SparkBackend
qf = Qualifire(
    backend=SparkBackend(spark),
    system_table=str(SYSTEM_DB),
    system_table_backend='sqlite',
    owner='complex-types-demo', bu='retail',
)
qf

## 1. Build a table with complex columns

`retail.events` has six columns:

- **`event_id`**: int — scalar, encodable everywhere.
- **`region`**: string, low cardinality — one-hot in shape/pattern.
- **`amount`**: double — pass-through numeric.
- **`tags`**: `array<string>` — list, complex.
- **`attrs`**: `map<string, string>` — dict, complex.
- **`address`**: `struct<city: string, zip: string>` — struct, complex.

In [ ]:
from pyspark.sql import Row
import random
random.seed(7)

spark.sql('CREATE DATABASE IF NOT EXISTS retail')
spark.sql('DROP TABLE IF EXISTS retail.events')

regions = ['us', 'uk', 'jp']
cities  = ['NYC', 'London', 'Tokyo', 'SF']

rows = []
for i in range(500):
    rows.append(Row(
        event_id=i,
        region=random.choice(regions),
        amount=round(random.gauss(100, 20), 2),
        tags=random.sample(['new', 'vip', 'promo', 'returning', 'gift'], k=random.randint(1, 3)),
        attrs={'channel': random.choice(['web', 'mobile', 'in-store']),
               'tier':    random.choice(['gold', 'silver', 'bronze'])},
        address=Row(city=random.choice(cities), zip=str(random.randint(10000, 99999))),
    ))
df = spark.createDataFrame(rows)
df.write.mode('overwrite').saveAsTable('retail.events')

spark.sql('DESCRIBE retail.events').show(truncate=False)
print()
spark.sql('SELECT * FROM retail.events LIMIT 3').show(truncate=False)

## 2. Profiling — complex columns silently skipped

`ProfilingCollector` classifies each column via `_SPARK_TYPE_MAP`
(`profiling/engine.py`). Anything not in the map (Array, Map,
Struct, Binary) is **silently skipped** with a debug log — the
profile result has zero stats for those columns.

**Key takeaway**: profiling assumes scalar columns. If you need
stats over a complex column, project it to a scalar form upstream
(`size(tags) AS tag_count`, `attrs['tier'] AS attr_tier`,
`address.city AS city`) and profile *that*.

Reference: [`docs/profiling.md`](../../docs/profiling.md) for the
single-pass profiling engine, the `_SPARK_TYPE_MAP` rules, and how
to set per-column stat selectors.

In [ ]:
from qualifire.core.config import (
    DatasetConfig, ProfilingCollectionConfig, QualifireConfig,
    ThresholdLevels, ThresholdRuleConfig, ThresholdValidationConfig,
)
from qualifire.core.context import QualifireContext
from qualifire.core.engine import QualifireEngine
from qualifire.core.exceptions import QualifireValidationError

# Profile every column in retail.events. We add a permissive threshold
# so the engine emits validation rows; collection rows for each
# scalar column carry the stats.
ds_profile = DatasetConfig(
    table='retail.events',
    description='Events fact — mixed scalar + complex columns.',
    validations=[ThresholdValidationConfig(
        name='profile_all_cols',
        description='Profiling collector run for diagnostic — no real bound.',
        collection=ProfilingCollectionConfig(
            columns=None,        # None → every column
            stats=['count', 'null_count', 'distinct_approx', 'min', 'max', 'mean'],
        ),
        rules=[ThresholdRuleConfig(
            metric='event_id.count',
            thresholds=ThresholdLevels(error={'min': 1}),
        )],
    )],
)
cfg = QualifireConfig(
    owner=qf.owner, bu=qf.bu, system_table=qf.system_table,
    datasets=[ds_profile],
)
engine = QualifireEngine(
    backend=qf.backend, storage=qf._storage,
    context=QualifireContext(), config=cfg,
)
try:
    result = engine.run()
except QualifireValidationError as e:
    result = e.result

# Collect emitted metric_names — these are the columns that survived
# the type filter (scalar-only).
profiled_columns = sorted({
    cr.metric_name.split('.', 1)[0]
    for ds in result.datasets for cr in ds.collection_results
    if '.' in (cr.metric_name or '')
})
all_columns = [f.name for f in spark.table('retail.events').schema.fields]
complex_columns = sorted(set(all_columns) - set(profiled_columns))

print('All columns         :', all_columns)
print('Profiled (kept)     :', profiled_columns)
print('Skipped (complex)   :', complex_columns)
print()
print('-- Sample stats per kept column --')
for ds in result.datasets:
    for cr in ds.collection_results[:12]:
        print(f'  {cr.metric_name:<30} = {cr.metric_value}')

## 3. Shape (Isolation Forest) — `drop_complex` flag

The shape validator samples records, converts to pandas, then runs
every column through `_encoding.py`. Default behaviour for complex
(unhashable dict/list, struct etc.):

- `drop_complex=False` (default): stringify the cell value
  (`str(<list>)`, `str(<dict>)`) and `LabelEncode` the resulting
  string column. Each unique JSON-shape becomes its own integer
  level. SHAP can then attribute, but feature names are the column
  name only — not the contents.
- `drop_complex=True`: drop the column entirely. Cleaner feature
  matrix; no spurious importance attached to columns the encoder
  can only stringify.

Pick `drop_complex=True` when complex columns are noise; pick the
default when their string-ified shape genuinely matters and you
expect the cardinality to be bounded.

Reference: [`docs/validators/shape.md`](../../docs/validators/shape.md)
for the encoding pipeline (numeric / boolean / categorical / complex),
tuning guide, and SHAP output shape.

In [ ]:
try:
    import sklearn  # noqa: F401
    has_sklearn = True
except ImportError:
    has_sklearn = False

if not has_sklearn:
    print('scikit-learn not installed — skip. `pip install scikit-learn shap`.')
else:
    def run_shape(drop_complex: bool, label: str):
        try:
            r = qf.validate(
                table='retail.events',
                validations=[Qualifire.shape_check(
                    name=f'shape_{label}',
                    description=f'Isolation Forest with drop_complex={drop_complex}.',
                    n_records=200,
                    past_dates=1, step='P1D',
                    n_estimators=50, contamination=0.1,
                    explain=True,
                    drop_complex=drop_complex,
                )],
            )
        except QualifireValidationError as e:
            r = e.result
        return r

    for keep_or_drop, label in [(False, 'keep_complex'), (True, 'drop_complex')]:
        r = run_shape(keep_or_drop, label)
        for ds in r.datasets:
            for vr in ds.validation_results:
                feats = (vr.details or {}).get('top_contributing_features') or []
                feature_names = [f['feature'] for f in feats]
                print(f'== drop_complex={keep_or_drop} ==')
                print(f'  severity : {vr.severity.value}')
                print(f'  message  : {vr.message[:160]}')
                if feats:
                    print(f'  features : {feature_names[:10]}')
                print()

## 4. Pattern (Random Forest) — same flag, same effect

Pattern check trains an RF classifier to distinguish current vs.
past samples and surfaces the columns that drove the split. It uses
the same `_encoding.py` module as shape, so `drop_complex` behaves
identically. Running both ways shows whether complex columns drove
the AUC — a high AUC that disappears when `drop_complex=True` is a
strong signal that the pattern lived in the complex columns'
string-shape distribution rather than the scalar fields.

Reference: [`docs/validators/pattern.md`](../../docs/validators/pattern.md)
for the leakage-control contract (you MUST list partition / date / ID
columns in `exclude_columns`), AUC interpretation, and tuning.

In [ ]:
if not has_sklearn:
    print('scikit-learn not installed — skip.')
else:
    def run_pattern(drop_complex: bool, label: str):
        try:
            r = qf.validate(
                table='retail.events',
                validations=[Qualifire.pattern_check(
                    name=f'pattern_{label}',
                    description=f'RF two-sample with drop_complex={drop_complex}.',
                    n_records=200,
                    past_dates=1, step='P1D',
                    n_estimators=50, cv_folds=2,
                    explain=True,
                    drop_complex=drop_complex,
                    exclude_columns=['event_id'],
                    thresholds={'warning': {'auc': 0.65}, 'error': {'auc': 0.85}},
                )],
            )
        except QualifireValidationError as e:
            r = e.result
        return r

    for keep_or_drop, label in [(False, 'keep_complex'), (True, 'drop_complex')]:
        r = run_pattern(keep_or_drop, label)
        for ds in r.datasets:
            for vr in ds.validation_results:
                feats = (vr.details or {}).get('top_contributing_features') or []
                feature_names = [f['feature'] for f in feats]
                print(f'== drop_complex={keep_or_drop} ==')
                print(f'  severity : {vr.severity.value}')
                print(f'  message  : {vr.message[:160]}')
                if feats:
                    print(f'  features : {feature_names[:10]}')
                print()

## Summary

| Place | Default | `drop_complex=True` | When to override |
|---|---|---|---|
| Profiling | Silently skip Array/Map/Struct/Binary | n/a | n/a — project to scalar form upstream if you need stats |
| Shape | Stringify + LabelEncode | Drop entirely | Set `True` when complex columns are noise; keep default when their *shape distribution* is meaningful |
| Pattern | Stringify + LabelEncode | Drop entirely | Same — and use the keep-vs-drop AUC delta as a diagnostic for whether the pattern lives in the complex columns |

**Rules of thumb**:

- If you want any stat on a complex column, **project it to a scalar form in your dataset query** (`size(tags)`, `attrs['tier']`, `address.city`). Profiling has no opinion on what's interesting inside a struct — that's domain-specific.
- If you're investigating an anomaly with shape/pattern and the top SHAP features are `tags` or `attrs`, that's the column-name only — the actual driver is some unique JSON-shape combination. Project to the meaningful scalar fields and re-run for interpretable attributions.
- Set `drop_complex=True` for cleaner experiments; it makes the feature matrix shape predictable and the SHAP attributions actionable.

## Cleanup

In [ ]:
import shutil
from pathlib import Path

spark.stop()
shutil.rmtree(WORK_DIR, ignore_errors=True)
for stray in ('metastore_db', 'derby.log'):
    p = Path.cwd() / stray
    if p.is_dir(): shutil.rmtree(p, ignore_errors=True)
    elif p.is_file(): p.unlink()
print('cleaned up:', WORK_DIR)